# 🚀 TravelMate AI - 11 Improved Itinerary Optimizer

Our enriched planning dataset is now available with 41 columns.

This notebook improves the baseline itinerary planner by making the day allocation more explicit.

## Goals

- Use the enriched travel dataset
- Respect the requested number of days
- Limit sightseeing time per day
- Balance high-scoring attractions across days
- Keep nearby places together when possible
- Use estimated visit duration
- Track estimated travel time
- Track estimated cost level
- Produce a readable day-by-day itinerary

### Important

The current duration and price fields are estimates created by our enrichment rules. They are useful for prototyping, but should be replaced by verified data before presenting the application as real travel guidance.


## 1. Imports

In [1]:
import math
import numpy as np
import pandas as pd

print("Libraries imported successfully! ✅")


Libraries imported successfully! ✅


## 2. Load the enriched planning dataset

In [2]:
df = pd.read_csv(
    "../data/processed/manali_travel_enriched.csv"
)

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (20, 41)


,place_id,name,address,country,latitude,longitude,rating,reviews,category,data_quality_score,...,travel_style,estimated_visit_minutes,estimated_price_level,is_outdoor,is_family_friendly,is_photography_friendly,is_adventure,estimated_best_time,planning_description,data_completeness
0,ChIJPeVowAaIBDkRnIfuPkskqi0,Hadimba Devi Temple,"Regency Road, Siyal Rd, Siyal, Manali, Himacha...",India,32.248353,77.181573,4.6,49688,Tourist attraction,6,...,"culture, nature, photography, shopping",60,2,0,0,1,0,morning,Hadimba Devi Temple is a temple experience in ...,1.0
1,ChIJ3VQyL0WHBDkR4Uml16nJto8,Old Manali snow point,"65XJ+J2W, Hadimba Temple Path, Old Manali, Man...",India,32.249112,77.180076,4.6,428,Tourist attraction,6,...,"adventure, culture, family, nature, photography",90,1,1,1,1,1,daytime,Old Manali snow point is a winter_experience e...,1.0
2,ChIJk9K3np2HBDkRS4FxsgYCMKI,Nehru Kund,"Bashisht, Himachal Pradesh 175103, India",India,32.285982,77.179824,4.4,7767,Tourist attraction,6,...,"culture, photography",60,1,0,0,1,0,daytime,Nehru Kund is a sightseeing experience in Mana...,1.0
3,ChIJJSBNlWOJBDkRkmyIuy0OvGE,Kullu Manali River rafting,"65VQ+7MF, Siyal, Manali, Himachal Pradesh 1751...",India,32.243187,77.189176,4.5,88,Tourist attraction,6,...,adventure,120,3,1,0,0,1,daytime,Kullu Manali River rafting is a rafting experi...,1.0
4,ChIJHZ3eboyHBDkRBLrRpkcXmO4,Jogini Falls,"On water fall way V.P.O.-Vashist 5 km.from, Ma...",India,32.275076,77.188146,4.6,10842,Tourist attraction,6,...,"culture, family, nature, photography",90,1,1,1,1,0,morning,Jogini Falls is a waterfall experience in Mana...,1.0


## 3. Select the columns needed for itinerary planning

In [3]:
planning_columns = [
    "place_id",
    "name",
    "latitude",
    "longitude",
    "rating",
    "reviews",
    "hybrid_score",
    "activity_type",
    "travel_style",
    "estimated_visit_minutes",
    "estimated_price_level",
    "estimated_best_time",
    "is_outdoor",
    "is_family_friendly",
    "is_photography_friendly",
    "is_adventure",
    "planning_description"
]

planning_df = df[planning_columns].copy()

planning_df.head()


,place_id,name,latitude,longitude,rating,reviews,hybrid_score,activity_type,travel_style,estimated_visit_minutes,estimated_price_level,estimated_best_time,is_outdoor,is_family_friendly,is_photography_friendly,is_adventure,planning_description
0,ChIJPeVowAaIBDkRnIfuPkskqi0,Hadimba Devi Temple,32.248353,77.181573,4.6,49688,0.463197,temple,"culture, nature, photography, shopping",60,2,morning,0,0,1,0,Hadimba Devi Temple is a temple experience in ...
1,ChIJ3VQyL0WHBDkR4Uml16nJto8,Old Manali snow point,32.249112,77.180076,4.6,428,0.451321,winter_experience,"adventure, culture, family, nature, photography",90,1,daytime,1,1,1,1,Old Manali snow point is a winter_experience e...
2,ChIJk9K3np2HBDkRS4FxsgYCMKI,Nehru Kund,32.285982,77.179824,4.4,7767,0.404494,sightseeing,"culture, photography",60,1,daytime,0,0,1,0,Nehru Kund is a sightseeing experience in Mana...
3,ChIJJSBNlWOJBDkRkmyIuy0OvGE,Kullu Manali River rafting,32.243187,77.189176,4.5,88,0.218735,rafting,adventure,120,3,daytime,1,0,0,1,Kullu Manali River rafting is a rafting experi...
4,ChIJHZ3eboyHBDkRBLrRpkcXmO4,Jogini Falls,32.275076,77.188146,4.6,10842,0.507617,waterfall,"culture, family, nature, photography",90,1,morning,1,1,1,0,Jogini Falls is a waterfall experience in Mana...


## 4. Validate planning data

We check the fields required by the optimizer.


In [4]:
required = [
    "name",
    "latitude",
    "longitude",
    "hybrid_score",
    "estimated_visit_minutes",
    "estimated_price_level"
]

print("Missing required values:")
print(planning_df[required].isnull().sum())


Missing required values:
name                       0
latitude                   0
longitude                  0
hybrid_score               0
estimated_visit_minutes    0
estimated_price_level      0
dtype: int64


## 5. Haversine distance

We estimate straight-line distance between two coordinates.

This is useful for a baseline optimizer but is **not the same as road distance**.


In [5]:
def haversine_km(lat1, lon1, lat2, lon2):
    earth_radius_km = 6371.0

    lat1 = math.radians(lat1)
    lon1 = math.radians(lon1)
    lat2 = math.radians(lat2)
    lon2 = math.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1)
        * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    c = 2 * math.atan2(
        math.sqrt(a),
        math.sqrt(1 - a)
    )

    return earth_radius_km * c


## 6. Create a distance matrix

In [6]:
n = len(planning_df)

coordinates = planning_df[
    ["latitude", "longitude"]
].to_numpy()

distance_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        distance_matrix[i, j] = haversine_km(
            coordinates[i, 0],
            coordinates[i, 1],
            coordinates[j, 0],
            coordinates[j, 1]
        )

print("Distance matrix shape:", distance_matrix.shape)


Distance matrix shape: (20, 20)


## 7. Estimate travel minutes

We use a configurable average speed.

Change this later when verified routing data becomes available.


In [7]:
AVERAGE_SPEED_KMPH = 25

travel_time_matrix = (
    distance_matrix / AVERAGE_SPEED_KMPH
) * 60


## 8. Normalize score components

We normalize hybrid score, rating and popularity so that they can be combined fairly.

Popularity uses `log1p(reviews)` because review counts span a very large range.


In [8]:
def minmax(series):
    series = series.astype(float)

    if series.max() == series.min():
        return pd.Series(
            np.ones(len(series)),
            index=series.index
        )

    return (
        (series - series.min())
        / (series.max() - series.min())
    )

planning_df["hybrid_norm"] = minmax(
    planning_df["hybrid_score"]
)

planning_df["rating_norm"] = minmax(
    planning_df["rating"]
)

planning_df["popularity_norm"] = minmax(
    np.log1p(
        planning_df["reviews"].clip(lower=0)
    )
)

planning_df[
    ["name", "hybrid_norm", "rating_norm", "popularity_norm"]
].head()


,name,hybrid_norm,rating_norm,popularity_norm
0,Hadimba Devi Temple,0.895894,0.777778,1.000000
1,Old Manali snow point,0.868059,0.777778,0.429428
2,Nehru Kund,0.758313,0.555556,0.777182
3,Kullu Manali River rafting,0.322953,0.666667,0.240583
4,Jogini Falls,1.000000,0.777778,0.817225


## 9. Create an itinerary priority score

This score is used only for selecting and ordering candidates.

```text
60% hybrid recommendation
25% rating
15% popularity
```


In [9]:
planning_df["priority_score"] = (
    0.60 * planning_df["hybrid_norm"]
    + 0.25 * planning_df["rating_norm"]
    + 0.15 * planning_df["popularity_norm"]
)

planning_df[
    ["name", "hybrid_score", "priority_score"]
].sort_values(
    "priority_score",
    ascending=False
).head(10)


,name,hybrid_score,priority_score
4,Jogini Falls,0.507617,0.917028
0,Hadimba Devi Temple,0.463197,0.881981
1,Old Manali snow point,0.451321,0.779694
6,Manali View Point,0.468197,0.774895
10,Kharma valley,0.406514,0.752582
14,Baror Parsha Waterfall,0.397755,0.734542
8,Lama Dugh Trek Start Point,0.423811,0.734447
5,Van Vihar National Park,0.447468,0.718082
18,Gulaba Viewpoint,0.397028,0.713766
2,Nehru Kund,0.404494,0.710454


## 10. Remove obvious weak candidates

The dataset contains a few generic or low-information results.

We flag them instead of silently deleting them.


In [10]:
weak_candidate_mask = (
    (planning_df["hybrid_score"] <= 0.12)
    | (planning_df["rating"] <= 4.0)
    | (
        planning_df["activity_type"].isin(
            ["sightseeing"]
        )
        & planning_df["travel_style"].eq("general")
    )
)

weak_candidates = planning_df[
    weak_candidate_mask
]

print("Weak candidate rows:", len(weak_candidates))

weak_candidates[[
    "name",
    "rating",
    "hybrid_score",
    "activity_type",
    "travel_style"
]]


Weak candidate rows: 1


,name,rating,hybrid_score,activity_type,travel_style
13,Himachal PARDESH,3.9,0.080937,sightseeing,general


## 11. Candidate pool

For this prototype, we keep the strongest 12 candidates unless the dataset is smaller.

This gives the optimizer enough choices to distribute across multiple days.


In [11]:
candidate_limit = min(12, len(planning_df))

candidates = (
    planning_df
    .sort_values(
        "priority_score",
        ascending=False
    )
    .head(candidate_limit)
    .copy()
)

candidates[[
    "name",
    "priority_score",
    "estimated_visit_minutes",
    "estimated_price_level"
]]


,name,priority_score,estimated_visit_minutes,estimated_price_level
4,Jogini Falls,0.917028,90,1
0,Hadimba Devi Temple,0.881981,60,2
1,Old Manali snow point,0.779694,90,1
6,Manali View Point,0.774895,45,1
10,Kharma valley,0.752582,120,1
14,Baror Parsha Waterfall,0.734542,90,1
8,Lama Dugh Trek Start Point,0.734447,150,1
5,Van Vihar National Park,0.718082,120,1
18,Gulaba Viewpoint,0.713766,45,1
2,Nehru Kund,0.710454,60,1


## 12. Itinerary settings

We explicitly request 3 days.

Each day has 7 hours of usable sightseeing time in this prototype.


In [12]:
NUM_DAYS = 3
DAY_START_MINUTES = 9 * 60
MAX_DAY_MINUTES = 7 * 60

print("Requested days:", NUM_DAYS)
print("Daily sightseeing budget:", MAX_DAY_MINUTES, "minutes")


Requested days: 3
Daily sightseeing budget: 420 minutes


## 13. Choose a balanced start for each day

Instead of always starting Day 2 and Day 3 with the globally best remaining place, we use a balanced selection from the remaining candidates.

This prevents one day from absorbing all high-priority places.


In [13]:
def choose_day_anchor(
    remaining_indices,
    used_day_scores
):
    ranked = sorted(
        remaining_indices,
        key=lambda idx: planning_df.loc[
            idx,
            "priority_score"
        ],
        reverse=True
    )

    if not ranked:
        return None

    # Prefer a strong candidate whose score is not
    # already represented heavily on another day.
    best = ranked[0]

    if len(ranked) > 1:
        for idx in ranked[:3]:
            score = planning_df.loc[
                idx,
                "priority_score"
            ]

            if all(
                abs(score - other) > 0.03
                for other in used_day_scores
            ):
                best = idx
                break

    return best


In [14]:
def build_day(
    available_indices,
    start_index,
    max_minutes
):
    route = []

    remaining = set(available_indices)

    current_index = start_index
    used_minutes = 0

    while remaining:

        feasible = []

        for idx in remaining:
            travel = (
                0
                if idx == current_index
                else travel_time_matrix[
                    planning_df.index.get_loc(current_index),
                    planning_df.index.get_loc(idx)
                ]
            )

            visit = planning_df.loc[
                idx,
                "estimated_visit_minutes"
            ]

            total = travel + visit

            if used_minutes + total <= max_minutes:
                feasible.append(
                    (idx, travel, visit, total)
                )

        if not feasible:
            break

        best_idx, travel, visit, total = max(
            feasible,
            key=lambda item: (
                planning_df.loc[
                    item[0],
                    "priority_score"
                ] / (1 + item[1])
            )
        )

        route.append({
            "index": best_idx,
            "travel_minutes": travel,
            "visit_minutes": visit
        })

        used_minutes += total
        remaining.remove(best_idx)
        current_index = best_idx

    return route


In [15]:
def format_time(minutes):
    minutes = int(round(minutes))

    hour = (minutes // 60) % 24
    minute = minutes % 60

    suffix = "AM" if hour < 12 else "PM"

    display_hour = hour % 12
    if display_hour == 0:
        display_hour = 12

    return f"{display_hour}:{minute:02d} {suffix}"


In [16]:
def generate_balanced_itinerary(
    candidates,
    num_days,
    max_day_minutes
):
    remaining = set(candidates.index)
    itinerary = []

    used_day_scores = []

    for day in range(1, num_days + 1):

        if not remaining:
            break

        anchor = choose_day_anchor(
            remaining,
            used_day_scores
        )

        if anchor is None:
            break

        day_route = build_day(
            remaining,
            anchor,
            max_day_minutes
        )

        if not day_route:
            break

        current_time = DAY_START_MINUTES
        day_rows = []

        for stop_number, stop in enumerate(
            day_route,
            start=1
        ):
            idx = stop["index"]

            arrival = (
                current_time
                + stop["travel_minutes"]
            )

            departure = (
                arrival
                + stop["visit_minutes"]
            )

            day_rows.append({
                "day": day,
                "stop": stop_number,
                "place": planning_df.loc[
                    idx,
                    "name"
                ],
                "activity_type": planning_df.loc[
                    idx,
                    "activity_type"
                ],
                "arrival": format_time(arrival),
                "departure": format_time(departure),
                "travel_before_minutes":
                    round(
                        stop["travel_minutes"],
                        1
                    ),
                "visit_minutes":
                    int(stop["visit_minutes"]),
                "estimated_price_level":
                    planning_df.loc[
                        idx,
                        "estimated_price_level"
                    ],
                "priority_score":
                    planning_df.loc[
                        idx,
                        "priority_score"
                    ],
                "latitude":
                    planning_df.loc[
                        idx,
                        "latitude"
                    ],
                "longitude":
                    planning_df.loc[
                        idx,
                        "longitude"
                    ]
            })

            current_time = departure

        itinerary.extend(day_rows)

        used_day_scores.append(
            planning_df.loc[
                anchor,
                "priority_score"
            ]
        )

        used_indices = [
            stop["index"]
            for stop in day_route
        ]

        remaining.difference_update(
            used_indices
        )

    return pd.DataFrame(itinerary)


In [17]:
itinerary_df = generate_balanced_itinerary(
    candidates,
    NUM_DAYS,
    MAX_DAY_MINUTES
)

itinerary_df


,day,stop,place,activity_type,arrival,departure,travel_before_minutes,visit_minutes,estimated_price_level,priority_score,latitude,longitude
0,1,1,Jogini Falls,waterfall,9:00 AM,10:30 AM,0.0,90,1,0.917028,32.275076,77.188146
1,1,2,Nehru Kund,sightseeing,10:33 AM,11:33 AM,3.5,60,1,0.710454,32.285982,77.179824
2,1,3,Hadimba Devi Temple,temple,11:44 AM,12:44 PM,10.0,60,2,0.881981,32.248353,77.181573
3,1,4,Old Manali snow point,winter_experience,12:44 PM,2:14 PM,0.4,90,1,0.779694,32.249112,77.180076
4,1,5,Manali Bazaar,shopping,2:16 PM,3:46 PM,2.5,90,2,0.562439,32.243991,77.189442
5,2,1,Manali View Point,viewpoint,9:00 AM,9:45 AM,0.0,45,1,0.774895,32.233835,77.187359
6,2,2,Van Vihar National Park,nature,9:46 AM,11:46 AM,1.5,120,1,0.718082,32.239113,77.189089
7,2,3,Lama Dugh Trek Start Point,trekking,11:51 AM,2:21 PM,4.1,150,1,0.734447,32.248903,77.175013
8,2,4,Gulaba Viewpoint,viewpoint,2:39 PM,3:24 PM,18.7,45,1,0.713766,32.317903,77.189340
9,3,1,Baror Parsha Waterfall,waterfall,9:00 AM,10:30 AM,0.0,90,1,0.734542,32.206785,77.184900


In [18]:
day_counts = (
    itinerary_df
    .groupby("day")
    .size()
    .reset_index(name="stops")
)

day_counts


,day,stops
0,1,5
1,2,4
2,3,3


In [19]:
if not itinerary_df.empty:

    total_travel = itinerary_df[
        "travel_before_minutes"
    ].sum()

    total_visit = itinerary_df[
        "visit_minutes"
    ].sum()

    total_estimated_cost_units = itinerary_df[
        "estimated_price_level"
    ].sum()

    average_priority = itinerary_df[
        "priority_score"
    ].mean()

    print(
        "Places scheduled:",
        len(itinerary_df)
    )

    print(
        "Total visit time:",
        round(total_visit, 1),
        "minutes"
    )

    print(
        "Estimated travel time:",
        round(total_travel, 1),
        "minutes"
    )

    print(
        "Estimated cost-level units:",
        total_estimated_cost_units
    )

    print(
        "Average priority score:",
        round(average_priority, 3)
    )


Places scheduled: 12
Total visit time: 1050 minutes
Estimated travel time: 78.7 minutes
Estimated cost-level units: 14
Average priority score: 0.741


In [20]:
daily_time = (
    itinerary_df
    .assign(
        total_minutes=lambda x:
            x["travel_before_minutes"]
            + x["visit_minutes"]
    )
    .groupby("day")["total_minutes"]
    .sum()
    .reset_index()
)

daily_time["within_budget"] = (
    daily_time["total_minutes"]
    <= MAX_DAY_MINUTES
)

daily_time


,day,total_minutes,within_budget
0,1,406.4,True
1,2,384.3,True
2,3,338.0,True


In [21]:
output_path = (
    "../data/processed/"
    "manali_improved_itinerary.csv"
)

itinerary_df.to_csv(
    output_path,
    index=False
)

print(f"✅ Saved: {output_path}")


✅ Saved: ../data/processed/manali_improved_itinerary.csv
